# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [18]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, find_all_hrs_ft_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    plot_ft_sync_alignment, plot_ft_trial_average,
    # Frequency Test interactive viewers
    make_ft_viewer, make_ft_sync_viewer, make_ft_avg_viewer,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    compute_h_comparison_data, plot_h_reflex_comparison,
    FT_SNAP_HZ, ft_snap_hz, ft_stim_adc_hz, compute_ft_trial_hz,
    load_all_recordings,
    make_viewer,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")


Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [19]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    

    #("HRPILOT-17 CM1",  "HRPilot-17_Control/BASELINE1_HRPILOT-17_BOOTH1_250US_10KHZ_8-20-26",        10000.0),
    #("HRPILOT-17 CM2",  "HRPilot-17_Control/BASELINE2_HRPILOT-17_BOOTH1_250US_10KHZ_8-21-26",        10000.0),
    #("HRPILOT-17 CM3",  "HRPilot-17_Control/BASELINE3_HRPILOT-17_BOOTH1_250US_10KHZ_8-24-26",        10000.0),
    #("HRPILOT-17 CM4",  "HRPilot-17_Control/BASELINE4_HRPILOT-17_BOOTH1_250US_10KHZ_8-25-26",        10000.0),
    #("HRPILOT-17 CM5",  "HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26",        10000.0),
    #("HRPILOT-17 CM6",  "HRPilot-17_Control/BASELINE6_HRPILOT-17_BOOTH1_250US_10KHZ_8-27-26",        10000.0),
    #("HRPILOT-17 CM7",  "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26",        10000.0),
    
    
    # Frequency Tests
    #("HRPILOT-17 FT1",  "HRPilot-17_Control/Old/CCC1_HRPILOT-17_BOOTH1_10KHZ_250US_8-5-26",        10000.0),
    #("HRPILOT-17 FT2",  "HRPilot-17_Control/Old/CCC2_HRPILOT-17_BOOTH1_10KHZ_250US_8-6-26",        10000.0),
    #("HRPILOT-17 Offline FT3",  "HRPilot-17_Control/Old/OFFLINE_FREQTEST1_HRPILOT-17_BOOTH1_10KHZ_250US_8-14-26",        10000.0),
    #("HRPILOT-17 Online FT4",  "HRPilot-17_Control/Old/ONLINE_FREQTEST1_HRPILOT-17_BOOTH1_10KHZ_250US_8-14-26",        10000.0),
        
    
        # HRPilot-23 Recordings
    #("HRPILOT-23 250US",  "Calibration/HRPilot-23/CALIB1_HRPILOT-23_BOOTH2_250US_10KHZ_8-31-26",        10000.0),
    #("HRPILOT-23 100US",  "Calibration/HRPilot-23/CALIB2_HRPILOT-23_BOOTH3_100US_10KHZ_9-2-26",        10000.0),
    #("FT1 HRPILOT-23 250US",  "Calibration/HRPilot-23/FT1_HRPILOT-23_100US_10KHZ_9-8-26",        10000.0),
    #("Calib3 HRPILOT-23 100US",  "Calibration/HRPilot-23/CALIB3_HRPILOT-23_BOOTH1_100US_10KHZ_9-11-26",        10000.0),
    
    #("CM3 FT OFfline HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_FT_OFFLINE_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
    #("CM4 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26",        10000.0),
    
    
      # HRPilot-23 Recordings
    
        
        # HRPilot-25 Recordings
        #("HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB1_HRPILOT-25_BOOTH2_250US_10KHZ_8-25-26",        10000.0),
        #("HRPILOT-25 100US",  "Calibration/HRPilot-25/CALIB2_HRPILOT-25_BOOTH2_100US_10KHZ_9-2-26",        10000.0),
        #("Calib3 HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB3_HRPILOT-25_BOOTH1_250US_10KHZ_9-10-26",        10000.0),
        #("Calib4 HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB4_HRPILOT-25_BOOTH2_100US_10KHZ_9-22-26",        10000.0),
    
            
        
        # HRPilot-33 Recordings
        ("Calib1 HRPILOT-33 250US",  "Calibration/HRPilot-33/CALIB1_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
        ("Calib1PT2 HRPILOT-33 250US",  "Calibration/HRPilot-33/CALIB1_PT2_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
        
        
        # HRPilot-34 Recordings
        ("Calib4 HRPILOT-34 250US",  "Calibration/HRPilot-34/CALIB4_FILTFILT_HRPILOT-34_BOOTH1_250US_10KHZ_9-23-26",        10000.0),
                
        
        
        # HRPilot-36 Recordings

        #("CM3 FT OFfline HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_FT_OFFLINE_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
        #("CM4 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26",        10000.0),

        
        #HRPilot-19 Recordings
        #("Calib1 HRPILOT-19 250US",  "Calibration/CALIB1_HRPILOT-19_BOOTH2_250US_10KHZ_9-18-26",        10000.0),
        
        #HRPilot-21 Recordings
        #("Calib1 HRPILOT-21 250US",  "Calibration/CALIB1_HRPILOT-21_BOOTH1_250US_10KHZ_9-18-26",        10000.0),
    
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

3 recording(s) configured.
  [0] 'Calib1 HRPILOT-33 250US'  →  Calibration/HRPilot-33/CALIB1_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26  (sample_rate=10000.0 Hz)
  [1] 'Calib1PT2 HRPILOT-33 250US'  →  Calibration/HRPilot-33/CALIB1_PT2_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26  (sample_rate=10000.0 Hz)
  [2] 'Calib4 HRPILOT-34 250US'  →  Calibration/HRPilot-34/CALIB4_FILTFILT_HRPILOT-34_BOOTH1_250US_10KHZ_9-23-26  (sample_rate=10000.0 Hz)


In [20]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
# Values within ±25% of a target snap to that target.
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

_all_recordings = load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM)
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'Calib1 HRPILOT-33 250US'  (Calibration/HRPilot-33/CALIB1_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26)
   .hrs1: 2829 trials  (MH Recruitment)
   .hrs2: not found
   .hrft: 187 trials across 1 file(s)
   .hrft Hz values (snapped): ['Single Pulse', '5.0 Hz', '10.0 Hz']
   App V3  |  Stages: ['mh_recruitment', 'frequency_test']  |  SR: 10000.0 Hz
   Settings sidecars: 2/2 stage(s)

── Loading: 'Calib1PT2 HRPILOT-33 250US'  (Calibration/HRPilot-33/CALIB1_PT2_HRPILOT-33_BOOTH1_250US_10KHZ_9-21-26)
   .hrs1: 2829 trials  (MH Recruitment)
   .hrs2: not found
   .hrft: 508 trials across 1 file(s)
   .hrft Hz values (snapped): ['Single Pulse', '5.0 Hz', '10.0 Hz']
   App V3  |  Stages: ['mh_recruitment', 'frequency_test']  |  SR: 10000.0 Hz

── Loading: 'Calib4 HRPILOT-34 250US'  (Calibration/HRPilot-34/CALIB4_FILTFILT_HRPILOT-34_BOOTH1_250US_10KHZ_9-23-26)
   .hrs1: 857 trials  (MH Recruitment)
   .hrs2: not found
   .hrft: 420 trials across 1 file(s)
   .hrft Hz values (snapped): ['Si

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [21]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'Calib1 HRPILOT-33 250US'
  MH Recruitment Curve (.hrs1): 2829 trials
  Frequency Test (.hrft): 187 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [22]:
# ── Recording & Stage Selector ─────────────────────────────────────────────────
# Change the active recording/stage here — all viewer sections update automatically.
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

# ── Shared viewer callback registry ────────────────────────────────────────────
if not isinstance(globals().get('_refresh_viewers'), dict):
    _refresh_viewers = {}

def _trigger_refresh():
    for _fn in list(_refresh_viewers.values()):
        try:
            _fn()
        except Exception as _err:
            import traceback
            print(f'[viewer refresh error] {_err}')
            traceback.print_exc()

# ── Active-recording state (module-level vars used by downstream cells) ────────
def _activate_recording(label):
    global _stage_map, recording_sample_rate, hrs1_header, \
           ft_trials, ft_header, ft_files, ACTIVE_STAGE, \
           _plot_trials, _plot_header, _plot_emg_blocks
    _rec = _all_recordings[label]
    _stage_map            = _rec['stage_map']
    recording_sample_rate = _rec['sample_rate']
    hrs1_header           = _rec['hrs1_header']
    ft_trials             = _rec.get('ft_trials')
    ft_header             = _rec.get('ft_header')
    ft_files              = _rec.get('ft_files', {})
    if globals().get('ACTIVE_STAGE') not in _stage_map:
        ACTIVE_STAGE = next(iter(_stage_map)) if _stage_map else None
    if ACTIVE_STAGE and ACTIVE_STAGE in _stage_map:
        _sel = _stage_map[ACTIVE_STAGE]
        _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]

_activate_recording(_active_rec_label)

# ── Global stage dropdown ──────────────────────────────────────────────────────
_sd_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
_stage_drop = Dropdown(options=_sd_opts, value=ACTIVE_STAGE,
                       description='Stage:', layout={'width': '480px'})

def _on_stage_change(change):
    global ACTIVE_STAGE, _plot_trials, _plot_header, _plot_emg_blocks
    ACTIVE_STAGE = _stage_drop.value
    _sel = _stage_map[ACTIVE_STAGE]
    _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]
    _trigger_refresh()

_stage_drop.observe(_on_stage_change, names='value')

if len(_all_recordings) > 1:
    _rec_drop = Dropdown(options=list(_all_recordings.keys()),
                         value=_active_rec_label,
                         description='Recording:', layout={'width': '640px'})

    def _on_rec_change(change):
        global _active_rec_label
        _active_rec_label = _rec_drop.value
        _activate_recording(_active_rec_label)
        _stage_drop.unobserve(_on_stage_change, names='value')
        _stage_drop.options = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
        _stage_drop.value   = ACTIVE_STAGE
        _stage_drop.observe(_on_stage_change, names='value')
        _trigger_refresh()

    _rec_drop.observe(_on_rec_change, names='value')
    _disp(VBox([_rec_drop, _stage_drop]))
else:
    _disp(_stage_drop)

# ── Status ─────────────────────────────────────────────────────────────────────
print(f'Active recording : {_active_rec_label!r}  (App V{_all_recordings[_active_rec_label]["app_version"]})')
print(f'Available stages ({len(_stage_map)}):')
for _k, (_t, _h, _e, _lbl) in _stage_map.items():
    _mark = '  ◄ active' if _k == ACTIVE_STAGE else ''
    print(f'  {_k!r:<22} → {_lbl}  ({len(_t)} trials){_mark}')
print(f'Sample rate : {recording_sample_rate} Hz')

# ── Helper: local Recording + Stage selector for any viewer ───────────────────
def _make_viewer_selector(refresh_fn):
    """Create independent Recording + Stage dropdowns for a viewer section.

    Returns (ctrl_widget, get_state_fn, sync_fn).

    get_state_fn() -> (stage_key, trials, header, emg, label, sample_rate, rec_label, hrs1_hdr)
    sync_fn()      -> called by the global selector to mirror its current state here.
    """
    from ipywidgets import Dropdown, VBox

    def _stage_opts(rec_label):
        sm = _all_recordings[rec_label]['stage_map']
        return [(lbl, sk) for sk, (_t, _h, _e, lbl) in sm.items() if _t]

    _local_rec   = [_active_rec_label]
    _local_stage = [ACTIVE_STAGE]

    _sd = Dropdown(options=_stage_opts(_active_rec_label),
                   value=ACTIVE_STAGE,
                   description='Stage:', layout={'width': '480px'})

    def _on_stage(change):
        _local_stage[0] = _sd.value
        refresh_fn()

    _sd.observe(_on_stage, names='value')

    if len(_all_recordings) > 1:
        _rd = Dropdown(options=list(_all_recordings.keys()),
                       value=_active_rec_label,
                       description='Recording:', layout={'width': '640px'})

        def _on_rec(change):
            _local_rec[0] = _rd.value
            _sd.unobserve(_on_stage, names='value')
            opts = _stage_opts(_rd.value)
            _sd.options = opts
            sm = _all_recordings[_rd.value]['stage_map']
            _sd.value = _local_stage[0] if _local_stage[0] in sm else (opts[0][1] if opts else None)
            _local_stage[0] = _sd.value
            _sd.observe(_on_stage, names='value')
            refresh_fn()

        _rd.observe(_on_rec, names='value')
        ctrl = VBox([_rd, _sd])
    else:
        _rd = None
        ctrl = VBox([_sd])

    def _get_state():
        rec = _local_rec[0]
        sk  = _local_stage[0]
        sm  = _all_recordings[rec]['stage_map']
        if sk not in sm:
            sk = next(iter(sm))
        st, sh, se, slbl = sm[sk]
        sr = _all_recordings[rec]['sample_rate']
        h1 = _all_recordings[rec]['hrs1_header']
        return sk, st, sh, se, slbl, sr, rec, h1

    def _sync():
        """Push global selector state into this viewer's local dropdowns."""
        if _rd is not None:
            _rd.unobserve(_on_rec, names='value')
            _rd.value = _active_rec_label
            _local_rec[0] = _active_rec_label
            _rd.observe(_on_rec, names='value')
        _sd.unobserve(_on_stage, names='value')
        opts = _stage_opts(_active_rec_label)
        _sd.options = opts
        sm = _all_recordings[_active_rec_label]['stage_map']
        _sd.value = ACTIVE_STAGE if ACTIVE_STAGE in sm else (opts[0][1] if opts else None)
        _local_stage[0] = _sd.value
        _sd.observe(_on_stage, names='value')
        refresh_fn()

    return ctrl, _get_state, _sync




Active recording : 'Calib1 HRPILOT-33 250US'  (App V3)
Available stages (2):
  'mh_recruitment'       → MH Recruitment Curve (.hrs1)  (2829 trials)  ◄ active
  'frequency_test'       → Frequency Test (.hrft)  (187 trials)
Sample rate : 10000.0 Hz


In [23]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
M_WAVE_START_MS = 1.8 
M_WAVE_END_MS   = 4.5
H_WAVE_START_MS = 6.5
H_WAVE_END_MS   = 9

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [24]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = True   # True → collapse every amplitude into one group
MERGED_GROUPS = []

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = True  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [25]:
# ── Pre-compute H-Reflex Comparison Data ──────────────────────────────────────────────
# Computed once here for instant rendering in the comparison plot below.
# Re-run this cell if you change PRE_AVG_MS, POST_AVG_MS, or H/M-wave window constants.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'H-reflex comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

H-reflex comparison data pre-computed for 3 recording(s), 2 stage(s): ['mh_recruitment', 'frequency_test']


# Section: Cross-Recording DataFrame & Calculations

`xr_df` is a flat per-trial DataFrame built from `_xr_cache`.  
Re-run this cell any time you re-run the pre-compute cell above.

In [26]:
import pandas as pd

# ── Build per-trial DataFrame from _xr_cache ──────────────────────────────────
_rows = []
for _rec_lbl, _stages in _xr_cache.items():
    for _sk, _d in _stages.items():
        _stage_lbl = _xr_stages.get(_sk, _sk)
        _h = _d['h_sizes']
        _m = _d['m_sizes']
        _bg = _d['bg_sizes']
        _n = max(len(_h), len(_m), len(_bg))
        for _i in range(_n):
            _hv  = float(_h[_i])  if _i < len(_h)  else float('nan')
            _mv  = float(_m[_i])  if _i < len(_m)  else float('nan')
            _bgv = float(_bg[_i]) if _i < len(_bg) else float('nan')
            _rows.append({
                'recording':  _rec_lbl,
                'stage':      _stage_lbl,
                'trial':      _i + 1,
                'h_size_uv':  _hv,
                'm_size_uv':  _mv,
                'bg_mra_uv':  _bgv,
                'hm_ratio':   _hv / _mv if (_mv and _mv > 0) else float('nan'),
            })

xr_df = pd.DataFrame(_rows)
print(f"xr_df: {len(xr_df)} rows  |  recordings: {xr_df['recording'].nunique()}  |  stages: {xr_df['stage'].nunique()}")
xr_df.head(10)

xr_df: 7630 rows  |  recordings: 3  |  stages: 2


,recording,stage,trial,h_size_uv,m_size_uv,bg_mra_uv,hm_ratio
0,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),1,21.801376,-28.095367,128.691971,NaN
1,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),2,187.816742,133.831894,50.514709,1.403378
2,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),3,-44.334736,12.663704,107.391953,-3.500930
3,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),4,99.120895,568.212357,103.567917,0.174443
4,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),5,94.267700,-165.433990,341.945221,NaN
5,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),6,22.882317,84.509941,122.773598,0.270765
6,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),7,-397.053665,-173.075439,616.616150,NaN
7,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),8,162.849274,309.070923,259.699524,0.526899
8,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),9,380.408039,826.717304,57.406353,0.460143
9,Calib1 HRPILOT-33 250US,MH Recruitment Curve (.hrs1),10,454.253143,846.744598,158.813019,0.536470


In [27]:
# ── Per-recording / per-stage summary statistics ─────────────────────────────
xr_summary = (
    xr_df.groupby(['recording', 'stage'])
    .agg(
        n_trials      = ('trial',      'count'),
        h_size_mean   = ('h_size_uv',  'mean'),
        h_size_std    = ('h_size_uv',  'std'),
        m_size_mean   = ('m_size_uv',  'mean'),
        m_size_std    = ('m_size_uv',  'std'),
        bg_mra_mean   = ('bg_mra_uv',  'mean'),
        hm_ratio_mean = ('hm_ratio',   'mean'),
        hm_ratio_std  = ('hm_ratio',   'std'),
    )
    .round(3)
)
xr_summary

n_trials  \
recording                  stage                                    
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)             187   
                           MH Recruitment Curve (.hrs1)      2829   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)             508   
                           MH Recruitment Curve (.hrs1)      2829   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)             420   
                           MH Recruitment Curve (.hrs1)       857   

                                                         h_size_mean  \
recording                  stage                                       
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)            691.657   
                           MH Recruitment Curve (.hrs1)      282.896   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)            664.486   
                           MH Recruitment Curve (.hrs1)      282.896   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)           -189.645   
                           MH Recruitment Curve (.hrs1)     -684.053   

                                                         h_size_std  \
recording                  stage                                      
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)           336.247   
                           MH Recruitment Curve (.hrs1)     255.587   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)           338.993   
                           MH Recruitment Curve (.hrs1)     255.587   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)           511.264   
                           MH Recruitment Curve (.hrs1)     727.936   

                                                         m_size_mean  \
recording                  stage                                       
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)           2736.866   
                           MH Recruitment Curve (.hrs1)     1661.499   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)           2643.909   
                           MH Recruitment Curve (.hrs1)     1661.499   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)            967.707   
                           MH Recruitment Curve (.hrs1)      897.937   

                                                         m_size_std  \
recording                  stage                                      
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)           862.957   
                           MH Recruitment Curve (.hrs1)     909.643   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)           638.876   
                           MH Recruitment Curve (.hrs1)     909.643   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)           697.347   
                           MH Recruitment Curve (.hrs1)    1620.905   

                                                         bg_mra_mean  \
recording                  stage                                       
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)            168.560   
                           MH Recruitment Curve (.hrs1)      146.921   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)            193.544   
                           MH Recruitment Curve (.hrs1)      146.921   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)            832.419   
                           MH Recruitment Curve (.hrs1)     1043.453   

                                                         hm_ratio_mean  \
recording                  stage                                         
Calib1 HRPILOT-33 250US    Frequency Test (.hrft)                0.252   
                           MH Recruitment Curve (.hrs1)          0.300   
Calib1PT2 HRPILOT-33 250US Frequency Test (.hrft)                0.267   
                           MH Recruitment Curve (.hrs1)          0.300   
Calib4 HRPILOT-34 250US    Frequency Test (.hrft)               -2.449   
                           MH Recruitment Curve (.hrs1)         -2.055   

                                          

In [28]:
# ── Custom calculations — edit this cell freely ───────────────────────────────
# Examples:

# Filter to a specific recording + stage
# rec = xr_df[(xr_df['recording'] == 'HRPILOT-17 FT1') & (xr_df['stage'] == 'Control Mode (.hrs2)')]

# Mean H-size per recording per stage (µV)
# xr_df.groupby(['recording', 'stage'])['h_size_uv'].mean()

# Coefficient of variation (%) for H-size per recording
# xr_df.groupby('recording')['h_size_uv'].agg(lambda x: x.std() / x.mean() * 100).rename('h_cv_pct')

# Export to CSV
# xr_df.to_csv('xr_data.csv', index=False)
# xr_summary.to_csv('xr_summary.csv')

print("xr_df and xr_summary are ready.  Edit this cell to run your calculations.")

xr_df and xr_summary are ready.  Edit this cell to run your calculations.


# Section 3e: Frequency Test Analysis (V3 FT)

Interactive viewer for `.hrsft` pulse-train data.  Layout mirrors the HRS2 Analysis viewer above.

**Navigation row** — `Prev` / `Next` / `Trial` dropdown navigate one trial at a time.  `Amp` dropdown filters to only trials at a specific stimulation amplitude (or "All").  `Freq` label shows the train frequency from the file header.

**Style row** — Controls how pulse traces are coloured and labelled:
- **Gradient** (default) — coolwarm colormap, blue = pulse 1 → red = last pulse.
- **Bold Ends** — same gradient, but first and last pulse are drawn thick so they stand out from the middle pulses.
- **Distinct** — tab20 qualitative palette, every pulse a unique colour (forces Labeled legend).
- **Show Legend** checkbox — toggle legend on/off.
- **Colorbar / Labeled** — gradient colorbar on the right axis, or individual per-pulse line entries in the plot legend.

**Pulse / zoom row** — `Pulse` dropdown highlights one specific pulse bold (lw 3.5, full alpha) and fades all others (lw 0.8, 15% alpha), making it easy to compare a single pulse against the rest of the train.  `Back to All` resets to equal weight.

**Controls row** — `Auto Y` checkbox auto-scales the y-axis; uncheck to type exact `Y min` / `Y max` bounds.  `Fig W` / `Fig H` resize the waveform figure (use when labels are getting clipped).  `Pre ms` / `Post ms` set the x-axis window around each pulse onset.

**Below the waveform viewer** — two static figures that update with every trial navigation:
- **H/M MRA Per Pulse** — mean rectified average (pre-stored app values) of the H-wave and M-wave per pulse across the train.  Shows homosynaptic (rate-dependent) depression.
- **H/M Peak Per Pulse** — max |EMG| within each wave window per pulse, computed from raw `trial_data`.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [29]:
# ── Frequency Test Analysis: Interactive Viewer ─────────────────────────────
from IPython.display import display as _disp
_disp(make_ft_viewer(
    _all_recordings, POST_PLOT_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
))

## Sync Alignment Diagnostic

Use `plot_ft_sync_alignment` to investigate whether low-frequency pulse trains show horizontal shifting due to clock drift between the stimulator and the recording system.

**Four panels:**
1. **ADC sync around nominal onset** — if the sync pulse slides left/right across pulses, the stimulator clock is drifting relative to the recording clock.
2. **EMG around detected (sync-corrected) onset** — should overlap cleanly after correction.
3. **EMG around nominal (clock-based) onset** — shows the raw uncorrected spread.
4. **Timing offset per pulse** — plots (detected − nominal) in ms; a sloped line = clock drift; digital-event markers shown as `|` ticks where available.

**Key parameters:**
- `trial` — pick a single FT trial from `ft_trials` (e.g. `ft_trials[0]`)
- `adc_threshold` — voltage threshold for sync-channel onset detection (default 4.5 V, match `STIM_ONSET_THRESHOLD`)
- `search_window_pct` — search ±this fraction of the inter-pulse period around each nominal onset (default 0.30 = ±30%)


In [30]:
# ── Sync Alignment Diagnostic: Interactive Viewer ─────────────────────────────
from IPython.display import display as _disp
_disp(make_ft_sync_viewer(
    _all_recordings,
    post_plot_ms=POST_PLOT_MS,
))


## Trial Averaging Viewer

Averages EMG responses **across trials at each pulse position**, analogous to Open Ephys's "Trial averaging" mode.

**What it shows:**
For N trials each containing K pulses, the viewer produces K traces — one per pulse position:
- **Bold colored trace (per pulse position)** — the cross-trial mean at that pulse position (pulse 1 averaged across all N trials, pulse 2 averaged across all N trials, etc.).  Colors run coolwarm: blue = pulse 1, red = last pulse.
- **Faint same-colored traces** (toggle with "Show individual") — each individual trial's contribution at that pulse position.  Lets you see trial-to-trial variability around the average.
- **Colorbar** — maps pulse number (1 → K) to the coolwarm color scale.
- **M/H wave shading** — same window constants as the rest of the notebook.

**Navigation:**
- **◀◀** / **▶▶** — jump to the first / last window position.
- **◀** / **▶** — slide the trial window backward / forward by 1 trial.
- **Reset** — return to trial 1.
- **N trials** — number of consecutive trials to include in the average window (default 5). Changing N resets to the first window.
- The **Trials X–Y of Z** label shows the current window position within the filtered trial list.

**Filters:**
- **Recording** — select which recording to display.
- **Freq** — filter to a specific pulse-train frequency (or "All").
- **Amp** — filter to a specific stimulation amplitude (or "All").

**Display controls:**
- `Show individual` — toggle the faint per-trial contribution traces on/off.
- `Pre ms` / `Post ms` — x-axis window around each pulse onset.
- `Fig W` / `Fig H` — figure dimensions in inches.
- `Auto Y` — auto-scale y-axis; uncheck to set exact `Y min` / `Y max`.


In [31]:
# ── Trial Averaging Viewer ─────────────────────────────────────────────────────
from IPython.display import display as _disp
_disp(make_ft_avg_viewer(
    _all_recordings,
    post_plot_ms=POST_PLOT_MS,
    m_start_ms=M_WAVE_START_MS,
    m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS,
    h_end_ms=H_WAVE_END_MS,
))
